In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1]))

import nltk
nltk.download("punkt_tab", quiet=True)

from src.data import load_ds, TEXT_COLUMNS
from src.preprocessing import (
    SPECIAL_TOKENS, PAD, UNK, SEP,
    clean_text, tokenize, remove_punctuation, preprocess,
    build_vocab, tokens_to_ids, build_embedding_matrix,
)

ds = load_ds()

In [2]:
samples = [
    "She walked into the woods.",
    "He didn’t   know what to do\nnext.",
    "It was Tom´s birthday — \"great!\"",
    "Broken \ufffd char, numbers 3.5 and 100%.",
]
for s in samples:
    c = clean_text(s)
    t = tokenize(c)
    print(repr(s))
    print("  clean   :", repr(c))
    print("  tokens  :", t)
    print("  no punct:", remove_punctuation(t))
    print()

'She walked into the woods.'
  clean   : 'She walked into the woods.'
  tokens  : ['She', 'walked', 'into', 'the', 'woods', '.']
  no punct: ['She', 'walked', 'into', 'the', 'woods']

'He didn’t   know what to do\nnext.'
  clean   : "He didn't know what to do next."
  tokens  : ['He', 'did', "n't", 'know', 'what', 'to', 'do', 'next', '.']
  no punct: ['He', 'did', "n't", 'know', 'what', 'to', 'do', 'next']

'It was Tom´s birthday — "great!"'
  clean   : 'It was Tom\'s birthday — "great!"'
  tokens  : ['It', 'was', 'Tom', "'s", 'birthday', '—', '``', 'great', '!', "''"]
  no punct: ['It', 'was', 'Tom', "'s", 'birthday', 'great']

'Broken � char, numbers 3.5 and 100%.'
  clean   : "Broken ' char, numbers 3.5 and 100%."
  tokens  : ['Broken', "'", 'char', ',', 'numbers', '3.5', 'and', '100', '%', '.']
  no punct: ['Broken', 'char', 'numbers', '3.5', 'and', '100']



In [3]:
for ex in ds["train"].select(range(3)):
    for col in TEXT_COLUMNS:
        print(f"{col:>14}: {ex[col]}")
        print(f"{'':>14}  {preprocess(ex[col])}")
    print()

 observation_1: A baby panda was lost in the woods.
                ['A', 'baby', 'panda', 'was', 'lost', 'in', 'the', 'woods']
 observation_2: He was reunited with his parents, who also lived at the zoo.
                ['He', 'was', 'reunited', 'with', 'his', 'parents', 'who', 'also', 'lived', 'at', 'the', 'zoo']
  hypothesis_1: He was never lost.
                ['He', 'was', 'never', 'lost']
  hypothesis_2: All the people from the zoo looked for him.
                ['All', 'the', 'people', 'from', 'the', 'zoo', 'looked', 'for', 'him']

 observation_1: A bank teller told me to rob the bank where she worked.
                ['A', 'bank', 'teller', 'told', 'me', 'to', 'rob', 'the', 'bank', 'where', 'she', 'worked']
 observation_2: We split the money equally.
                ['We', 'split', 'the', 'money', 'equally']
  hypothesis_1: The mountain told me when no one would be there.
                ['The', 'mountain', 'told', 'me', 'when', 'no', 'one', 'would', 'be', 'there']
  hypothes

In [4]:
import pandas as pd
from collections import Counter

train_df = ds["train"].to_pandas()
texts = pd.concat([train_df[c] for c in TEXT_COLUMNS], ignore_index=True)
cleaned = texts.map(clean_text)

checks = {
    "non-ascii": r"[^\x00-\x7f]",
    "U+FFFD": "\ufffd",
    "double space": r"\s{2,}",
    "leading/trailing ws": r"^\s|\s$",
}
for name, pat in checks.items():
    hits = cleaned[cleaned.str.contains(pat)]
    print(f"{name:>20}: {len(hits)}", list(hits.head(3)))

non_ascii = Counter(ch for t in cleaned for ch in t if ord(ch) > 127)
non_ascii.most_common(20)

           non-ascii: 2 ['His ex-girlfriend was inside the café.', 'Jerry chickened out of signing up for an open mic night at the local café.']
              U+FFFD: 0 []
        double space: 0 []
 leading/trailing ws: 0 []


[('é', 2)]

In [5]:
train_tokens = [preprocess(t) for t in texts]
print("texts with no tokens left:", [t for t, toks in zip(texts, train_tokens) if not toks])
pd.Series([len(t) for t in train_tokens]).describe()

texts with no tokens left: []


count    67180.000000
mean         8.657115
std          2.909775
min          1.000000
25%          7.000000
50%          8.000000
75%         10.000000
max         30.000000
dtype: float64

In [6]:
from src.embeddings import load_word2vec

wv = load_word2vec()
len(wv), wv.vector_size

(3000000, 300)

In [7]:
vocab = build_vocab(train_tokens, wv)

counts = Counter(t for toks in train_tokens for t in toks)
oov = Counter({t: n for t, n in counts.items() if t not in vocab})
print("vocab size:", len(vocab), "| first entries:", list(vocab)[:8])
print(f"OOV types : {len(oov)} / {len(counts)}")
print(f"OOV tokens: {sum(oov.values()) / sum(counts.values()):.2%}")
oov.most_common(30)

vocab size: 16833 | first entries: ['<pad>', '<unk>', '<sep>', 'A', 'baby', 'panda', 'was', 'lost']
OOV types : 745 / 17575
OOV tokens: 1.02%


[("'s", 4814),
 ('#th', 30),
 ('co-worker', 23),
 ('#nd', 12),
 ('co-workers', 10),
 ('#st', 9),
 ('selfie', 8),
 ('mTurk', 8),
 ('t-shirt', 7),
 ('after-school', 7),
 ('ex-boyfriend', 7),
 ('mturk', 6),
 ('PS#', 6),
 ('#K', 6),
 ('part-time', 6),
 ('A+', 6),
 ('Antonela', 5),
 ('mother-in-law', 5),
 ('two-year', 5),
 ('Fariah', 5),
 ('Mturk', 5),
 ('rear-ended', 5),
 ('Shalil', 4),
 ('#k', 4),
 ('sister-in-law', 4),
 ('Marisse', 4),
 ('#rd', 4),
 ('take-out', 4),
 ('Padfoot', 4),
 ('BunnBunn', 4)]

In [8]:
toks = preprocess(ds["validation"][0]["observation_1"])
print(list(zip(toks, tokens_to_ids(toks, vocab))))

val_df = ds["validation"].to_pandas()
val_tokens = [t for c in TEXT_COLUMNS for text in val_df[c] for t in preprocess(text)]
print(f"validation <unk> rate: {sum(t not in vocab for t in val_tokens) / len(val_tokens):.2%}")

[('I', 162), ('went', 169), ('to', 15), ('Baltimore', 1616), ('in', 8), ('####', 1111), ('to', 15), ('audition', 6996), ('for', 114), ('music', 1071), ('school', 113)]
validation <unk> rate: 2.84%


In [9]:
import numpy as np

E = build_embedding_matrix(vocab, wv)
print("shape:", E.shape, E.dtype)
print("pad all zero:", not E[vocab[PAD]].any(), "| unk all zero:", not E[vocab[UNK]].any())
print("sep norm:", np.linalg.norm(E[vocab[SEP]]).round(3))
w = list(vocab)[len(SPECIAL_TOKENS)]
print(f"row for {w!r} matches wv:", np.allclose(E[vocab[w]], wv[w]))
print("zero rows besides pad/unk:", int((~E.any(axis=1)).sum()) - 2)

shape: (16833, 300) float32
pad all zero: True | unk all zero: True
sep norm: 1.61
row for 'A' matches wv: True
zero rows besides pad/unk: 0
